In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("ronantakizawa/japanese-honorifics")

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/ronantakizawa/japanese-honorifics/resolve/95f53e628f390e02ae804ab49a10530a7ddb8d5e/japanese-honorifics.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since ronantakizawa/japanese-honorifics couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\doria\.cache\huggingface\datasets\ronantakizawa___japanese-honorifics\default\0.0.0\95f53e628f390e02ae804ab49a10530a7ddb8d5e (last modified on Fri Oct  2 12:12:15 2026).


In [28]:
# Convert the 'train' split to a pandas DataFrame safely
data_0 = ds['train'].to_pandas()

# Verify it worked by viewing the first 5 rows
data_0.head()


,base_sentence,teineigo,sonkeigo,kenjogo,english_translation
0,食事を食べる。,ご食事を食べます。,ご食事を召し上がります。,ご食事をいただきます。,I eat the meal.
1,本を読む。,本を読みます。,本をお読みになります。,本を拝読します。,I read a book.
2,昨日の会議で新しいプロジェクトについて話した。,昨日の会議で新しいプロジェクトについて話しました。,昨日の会議で新しいプロジェクトについてお話しになりました。,昨日の会議で新しいプロジェクトについてお話しいたしました。,Talked about the new project at yesterday's me...
3,質問をする。,質問をします。,ご質問されます。,お伺いします。,To ask a question.
4,彼を待つ。,彼を待ちます。,彼をお待ちになります。,彼をお待ちいたします。,Wait for him.


In [31]:
data_0.shape

(115, 5)

In [7]:
data_1 = pd.read_csv("../raw/annot_data_1.csv")
data_2 = pd.read_csv("../raw/annot_data_2.csv")

In [10]:
data_1.head()

,元文,変換文,変換ラベル
0,朝ごはんはトーストにバターとべジマイトを薄くぬって食べました。,朝ごはんはトーストにバターとべジマイトを薄くぬっていただきました。,変換:謙譲語
1,そして10時くらいに、喫茶店でレーシャルとジョノサンとベルに会いました。,そして10時くらいに、喫茶店でレーシャルとジョノサンとベルに会いました。,無変換
2,24日から26日まではavconというアニメとテレビゲームの大会です。,24日から26日まではavconというアニメとテレビゲームの大会です。,無変換
3,材料はひよこ豆とパスタと卵とニンニクと小麦粉といろいろなスパイスです。,材料はひよこ豆とパスタと卵とニンニクと小麦粉といろいろなスパイスです。,無変換
4,そして、ニンジンとキュウリとほうれん草と（日本の）マヨネーズもあります。,そして、ニンジンとキュウリとほうれん草と（日本の）マヨネーズもあります。,無変換


In [11]:
data_2.head()

,元文,変換文,変換ラベル
0,朝ごはんはトーストにバターとべジマイトを薄くぬって食べました。,朝ごはんはトーストにバターとべジマイトを薄くぬって食べました。,無変換
1,そして10時くらいに、喫茶店でレーシャルとジョノサンとベルに会いました。,そして10時くらいに、喫茶店でレーシャルとジョノサンとベルに会いました。,無変換
2,24日から26日まではavconというアニメとテレビゲームの大会です。,24日から26日まではavconというアニメとテレビゲームの大会です。,無変換
3,材料はひよこ豆とパスタと卵とニンニクと小麦粉といろいろなスパイスです。,材料はひよこ豆とパスタと卵とニンニクと小麦粉といろいろなスパイスです。,無変換
4,そして、ニンジンとキュウリとほうれん草と（日本の）マヨネーズもあります。,そして、ニンジンとキュウリとほうれん草と（日本の）マヨネーズもあります。,無変換


In [29]:
total_rows = len(data_1) + len(data_2) + len(data_0)
print(f"Shape: {total_rows}")

Shape: 895


In [48]:
def standardize_label(label):
    if pd.isna(label):
        return 'unknown'
    
    # Standard string replacements
    s = str(label)
    
    # Prefix translations
    s = s.replace('変換:', '')
    s = s.replace('どちらでも良い:', 'optional_')
    s = s.replace('無変換', 'no_change')
    
    # Keigo type translations
    s = s.replace('丁寧語', 'teineigo')
    s = s.replace('尊敬語', 'sonkeigo')
    s = s.replace('謙譲語', 'kenjogo')
    
    # Separator translations
    s = s.replace('、', '_and_')
    s = s.replace('Um', 'unclear')
    
    return s.strip()

# 1. Unpivot data_0 into long format (1 row -> 3 rows)
data_0_long = pd.melt(
    data_0,
    id_vars=['base_sentence'],
    value_vars=['teineigo', 'sonkeigo', 'kenjogo'],
    var_name='transformation_label',
    value_name='transformed_text'
)

# Standardize base_sentence column to source_text
data_0_long = data_0_long.rename(columns={'base_sentence': 'source_text'})
data_0_long = data_0_long[['source_text', 'transformed_text', 'transformation_label']]
data_0_long = data_0_long.dropna(subset=['transformed_text'])

# 2. Rename Japanese columns in data_1 and data_2 to English
column_map = {
    '元文': 'source_text',
    '変換文': 'transformed_text',
    '変換ラベル': 'transformation_label'
}

data_1_en = data_1.rename(columns=column_map)
data_2_en = data_2.rename(columns=column_map)

# 3. Convert Japanese keigo labels in data_1 and data_2 into Romaji / English
label_map = {
    '変換:丁寧語': 'teineigo',
    '変換:尊敬語': 'sonkeigo',
    '変換:謙譲語': 'kenjogo',
    '無変換': 'no_change'
}

data_1_en['transformation_label'] = data_1_en['transformation_label'].apply(standardize_label)
data_2_en['transformation_label'] = data_2_en['transformation_label'].apply(standardize_label)

# 4. Combine all datasets and drop exact duplicates
df_combined = pd.concat([data_0_long, data_1_en, data_2_en], ignore_index=True)
df_combined = df_combined.drop_duplicates().reset_index(drop=True)

In [49]:
df_combined.head(10)

,source_text,transformed_text,transformation_label
0,食事を食べる。,ご食事を食べます。,teineigo
1,本を読む。,本を読みます。,teineigo
2,昨日の会議で新しいプロジェクトについて話した。,昨日の会議で新しいプロジェクトについて話しました。,teineigo
3,質問をする。,質問をします。,teineigo
4,彼を待つ。,彼を待ちます。,teineigo
5,プロジェクトを完了する。,プロジェクトを完了します。,teineigo
6,そのことを考える。,そのことを考えます。,teineigo
7,彼の考えがわかる。,彼の考えがわかります。,teineigo
8,彼に本をあげた。,彼に本をあげました。,teineigo
9,プレゼントをもらった。,プレゼントをもらいました。,teineigo


In [50]:
df_combined.to_csv('integrated_csv.csv', index=False)